# mt_figures — Thesis Visualizations

All charts and figures for the experiment comparison. Loaded by `00_main` via `%run ./mt_figures`.

**Requires:** `summary_df` and `dynamic_filtered_results_df` from the experiment run.

## Contents
Thesis Figures (8 publication-ready panels: 7 quality + 1 abstention)

In [0]:
# ============================================================
# THESIS FIGURES — 7 Publication-Ready Panels
# ============================================================
# Requires: summary_df (from experiment summary cell)
# Produces: 7 matplotlib figures for thesis chapter
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import pandas as pd

if 'summary_df' not in dir() or len(summary_df) == 0:
    print("No summary_df available. Run the experiment first.")
else:
    # --- SETUP: column resolution (alias legacy names if needed) ---
    # Use answerable_df for quality figures; unanswerable_df for abstention figure
    _df = answerable_df.copy() if 'answerable_df' in dir() and len(answerable_df) > 0 else summary_df.copy()
    _unans_df = unanswerable_df.copy() if 'unanswerable_df' in dir() and len(unanswerable_df) > 0 else pd.DataFrame()
    _quality_label = "answerable only" if 'answerable_df' in dir() and len(answerable_df) > 0 else "all questions"
    _col_map = {'claim_precision': 'answer_precision', 'claim_recall': 'answer_recall',
                'claim_f1': 'answer_f1', 'groundedness': 'claim_groundedness',
                'hallucination_rate': 'numeric_hallucination_risk'}
    for _new, _old in _col_map.items():
        if _new not in _df.columns and _old in _df.columns:
            _df[_new] = _df[_old]
    if 'answer_correct' not in _df.columns:
        _df['answer_correct'] = (_df.get('claim_f1', _df.get('answer_f1', pd.Series(0, index=_df.index))) > 0)

    _arch_order = [a for a in ['SAS', 'SAS_RAG', 'MAS_RAG', 'DYNAMIC_FILTERED_MAS_RAG'] if a in _df['architecture'].unique()]
    _arch_short = {'SAS': 'SAS', 'SAS_RAG': 'SAS_RAG', 'MAS_RAG': 'MAS_RAG', 'DYNAMIC_FILTERED_MAS_RAG': 'DYN_FILT'}
    _diff_order = [d for d in ['easy', 'medium', 'hard', 'very_hard'] if d in _df['difficulty'].values]
    _colors = {'SAS': '#1f77b4', 'SAS_RAG': '#ff7f0e', 'MAS_RAG': '#2ca02c', 'DYNAMIC_FILTERED_MAS_RAG': '#d62728'}
    _diff_colors = {'easy': '#2ecc71', 'medium': '#f39c12', 'hard': '#e74c3c', 'very_hard': '#8e44ad'}

    def _bootstrap_ci(data, n_boot=1000, ci=0.95):
        """Bootstrap 95% CI for the mean."""
        data = data.dropna().values
        if len(data) < 2:
            m = np.mean(data) if len(data) else 0
            return m, m, m
        rng = np.random.default_rng(42)
        means = [np.mean(rng.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
        lo = np.percentile(means, (1 - ci) / 2 * 100)
        hi = np.percentile(means, (1 + ci) / 2 * 100)
        return np.mean(data), lo, hi

    # ==================================================================
    # FIGURE 1: Quality by Difficulty (Precision, Recall, F1)
    # ==================================================================
    _n_arch = len(_arch_order)
    _offsets = np.linspace(-0.28, 0.28, _n_arch)

    fig1, axes1 = plt.subplots(1, 3, figsize=(18, 8), sharey=True)
    fig1.suptitle(f'Figure 1: Answer Quality by Difficulty ({_quality_label})', fontsize=13, y=1.02)

    for ax, (metric, label) in zip(axes1, [('claim_precision', 'Claim Precision'),
                                            ('claim_recall', 'Claim Recall'),
                                            ('claim_f1', 'Claim F1')]):
        _col = metric if metric in _df.columns else _col_map.get(metric, metric)
        for i, arch in enumerate(_arch_order):
            for j, diff in enumerate(_diff_order):
                subset = _df[(_df['architecture'] == arch) & (_df['difficulty'] == diff)][_col]
                if len(subset) == 0:
                    continue
                mean, lo, hi = _bootstrap_ci(subset)
                x_pos = j + _offsets[i]
                ax.errorbar(x_pos, mean, yerr=[[mean - lo], [hi - mean]], fmt='o', color=_colors[arch],
                            markersize=10, capsize=6, capthick=2, linewidth=2, zorder=5)
                jitter = np.random.default_rng(i * 10 + j).uniform(-0.05, 0.05, size=len(subset))
                ax.scatter(np.full(len(subset), x_pos) + jitter, subset.values, color=_colors[arch],
                           alpha=0.35, s=28, zorder=3)
        ax.set_xticks(range(len(_diff_order)))
        ax.set_xticklabels(_diff_order, fontsize=10)
        ax.set_xlim(-0.5, len(_diff_order) - 0.5)
        ax.set_ylim(-0.02, 1.05)
        ax.set_title(label, fontsize=12)
        ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
    axes1[0].set_ylabel('Score (0\u20131)', fontsize=11)
    _legend_handles = [mpatches.Patch(color=_colors[a], label=_arch_short[a]) for a in _arch_order]
    fig1.legend(handles=_legend_handles, fontsize=10, loc='upper center', ncol=_n_arch,
               bbox_to_anchor=(0.5, 1.0), frameon=True, edgecolor='grey')
    fig1.tight_layout(rect=[0, 0, 1, 0.94])
    display(fig1)
    plt.close(fig1)

    # ==================================================================
    # FIGURE 2: Quality × Difficulty Interaction Plot
    # ==================================================================
    fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
    fig2.suptitle('Figure 2: Performance by Difficulty', fontsize=12, y=1.01)

    for metric, ax, ylabel, title in [('claim_precision', axes2[0,0], 'Mean Claim Precision', 'Precision by Difficulty'),
                                       ('claim_recall', axes2[0,1], 'Mean Claim Recall', 'Recall by Difficulty'),
                                       ('claim_f1', axes2[1,0], 'Mean Claim F1', 'F1 by Difficulty'),
                                       ('total_tokens', axes2[1,1], 'Mean Total Tokens', 'Tokens by Difficulty')]:
        _col = metric if metric in _df.columns else _col_map.get(metric, metric)
        for arch in _arch_order:
            means, los, his, xs = [], [], [], []
            for j, diff in enumerate(_diff_order):
                subset = _df[(_df['architecture'] == arch) & (_df['difficulty'] == diff)][_col]
                if len(subset) == 0:
                    continue
                m, lo, hi = _bootstrap_ci(subset)
                means.append(m); los.append(lo); his.append(hi); xs.append(j)
            ax.plot(xs, means, 'o-', color=_colors[arch], label=_arch_short[arch], linewidth=2, markersize=7)
            ax.fill_between(xs, los, his, alpha=0.12, color=_colors[arch])
        ax.set_xticks(range(len(_diff_order)))
        ax.set_xticklabels(_diff_order, fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=9)
    fig2.tight_layout()
    display(fig2)
    plt.close(fig2)

    # ==================================================================
    # FIGURE 3: Paired Difference Plot (ΔF1, DYNAMIC vs MAS_RAG)
    # ==================================================================
    if 'MAS_RAG' in _df['architecture'].values and 'DYNAMIC_FILTERED_MAS_RAG' in _df['architecture'].values:
        _mas = _df[_df['architecture'] == 'MAS_RAG'].set_index('question_id')
        _dyn = _df[_df['architecture'] == 'DYNAMIC_FILTERED_MAS_RAG'].set_index('question_id')
        _common = _mas.index.intersection(_dyn.index)

        _f1_col = 'claim_f1' if 'claim_f1' in _mas.columns else 'answer_f1'
        _hal_col = 'hallucination_rate' if 'hallucination_rate' in _mas.columns else 'numeric_hallucination_risk'

        _deltas = pd.DataFrame({
            'question_id': _common,
            'difficulty': [_mas.loc[q, 'difficulty'] for q in _common],
            'delta_f1': [(_dyn.loc[q, _f1_col] or 0) - (_mas.loc[q, _f1_col] or 0) for q in _common],
            'delta_halluc': [(_mas.loc[q, _hal_col] or 0) - (_dyn.loc[q, _hal_col] or 0) for q in _common],
            'token_saving_pct': [((_mas.loc[q, 'total_tokens'] - _dyn.loc[q, 'total_tokens']) / max(_mas.loc[q, 'total_tokens'], 1)) * 100 for q in _common],
            'latency_saving_pct': [((_mas.loc[q, 'latency_seconds'] - _dyn.loc[q, 'latency_seconds']) / max(_mas.loc[q, 'latency_seconds'], 0.1)) * 100 for q in _common],
        }).sort_values('difficulty', key=lambda x: x.map({d: i for i, d in enumerate(_diff_order)}))

        fig3, axes3 = plt.subplots(2, 2, figsize=(15, 10))
        fig3.suptitle('Figure 3: Paired Differences (DYNAMIC vs MAS_RAG, per question)', fontsize=12, y=1.01)

        for ax, (col, title, ylabel) in zip(axes3.flat, [
            ('delta_f1', '\u0394 Claim F1', 'F1(DYNAMIC) \u2212 F1(MAS)'),
            ('delta_halluc', 'Hallucination Reduction', 'Halluc(MAS) \u2212 Halluc(DYN)'),
            ('token_saving_pct', 'Token Savings %', '(MAS\u2212DYN)/MAS \u00d7 100'),
            ('latency_saving_pct', 'Latency Savings %', '(MAS\u2212DYN)/MAS \u00d7 100'),
        ]):
            colors = [_diff_colors.get(d, 'grey') for d in _deltas['difficulty']]
            ax.bar(range(len(_deltas)), _deltas[col], color=colors, edgecolor='black', linewidth=0.5, alpha=0.8)
            ax.axhline(0, color='black', linewidth=1)
            ax.set_xticks(range(len(_deltas)))
            ax.set_xticklabels(_deltas['question_id'], rotation=45, ha='right', fontsize=8)
            ax.set_ylabel(ylabel, fontsize=9)
            ax.set_title(title, fontsize=11)
        _legend_patches = [mpatches.Patch(color=_diff_colors[d], label=d) for d in _diff_order]
        axes3[0, 1].legend(handles=_legend_patches, fontsize=9, loc='upper right')
        fig3.tight_layout()
        display(fig3)
        plt.close(fig3)
    else:
        print("Figure 3 skipped: need both MAS_RAG and DYNAMIC_FILTERED_MAS_RAG.")

    # ==================================================================
    # FIGURE 4: Quadrant Plot (Token Savings vs ΔF1)
    # ==================================================================
    if '_deltas' in dir():
        fig4, ax4 = plt.subplots(figsize=(9, 7))
        fig4.suptitle('Figure 4: Efficiency\u2013Quality Trade-off (per question)', fontsize=12)
        for diff in _diff_order:
            sub = _deltas[_deltas['difficulty'] == diff]
            ax4.scatter(sub['token_saving_pct'], sub['delta_f1'], color=_diff_colors[diff],
                        s=80, edgecolors='black', linewidth=0.6, label=diff, zorder=5)
        ax4.axhline(0, color='black', linewidth=0.8, linestyle='-')
        ax4.axvline(0, color='black', linewidth=0.8, linestyle='-')
        ax4.text(0.95, 0.95, 'DOMINATES\n(+quality, +efficiency)', transform=ax4.transAxes,
                 ha='right', va='top', fontsize=8, color='green', alpha=0.7)
        ax4.text(0.05, 0.95, '+quality\n\u2212efficiency', transform=ax4.transAxes,
                 ha='left', va='top', fontsize=8, color='orange', alpha=0.7)
        ax4.text(0.95, 0.05, '\u2212quality\n+efficiency', transform=ax4.transAxes,
                 ha='right', va='bottom', fontsize=8, color='orange', alpha=0.7)
        ax4.text(0.05, 0.05, 'INFERIOR\n(\u2212quality, \u2212efficiency)', transform=ax4.transAxes,
                 ha='left', va='bottom', fontsize=8, color='red', alpha=0.7)
        ax4.set_xlabel('Token Saving % (positive = DYNAMIC uses fewer)', fontsize=10)
        ax4.set_ylabel('\u0394F1 (positive = DYNAMIC is better)', fontsize=10)
        ax4.legend(title='Difficulty', fontsize=9)
        ax4.grid(True, alpha=0.3)
        fig4.tight_layout()
        display(fig4)
        plt.close(fig4)

        _q1 = ((_deltas['token_saving_pct'] > 0) & (_deltas['delta_f1'] > 0)).sum()
        _q2 = ((_deltas['token_saving_pct'] <= 0) & (_deltas['delta_f1'] > 0)).sum()
        _q3 = ((_deltas['token_saving_pct'] > 0) & (_deltas['delta_f1'] <= 0)).sum()
        _q4 = ((_deltas['token_saving_pct'] <= 0) & (_deltas['delta_f1'] <= 0)).sum()
        print(f"  Quadrant counts: Dominates={_q1}, +Qual/-Eff={_q2}, -Qual/+Eff={_q3}, Inferior={_q4}")

    # ==================================================================
    # FIGURE 5: Outcome Composition (Stacked Bar)
    # ==================================================================
    fig5, ax5 = plt.subplots(figsize=(10, 5))
    fig5.suptitle('Figure 5: Outcome Composition per Architecture', fontsize=12)

    _outcome_data = []
    for arch in _arch_order:
        adf = _df[_df['architecture'] == arch]
        n = len(adf)
        _correct = ((adf.get('claim_f1', adf.get('answer_f1', pd.Series(0, index=adf.index)))) > 0).sum()
        _sql_fail = (adf['sql_execution_status'] == 'failed').sum()
        _abstain_phrases = ['CANNOT_ANSWER', 'INSUFFICIENT_EVIDENCE', 'ABSTAIN']
        _abstained = adf['generated_answer'].apply(
            lambda x: any(p in str(x).upper() for p in _abstain_phrases) if pd.notna(x) else True
        ).sum() if 'generated_answer' in adf.columns else 0
        _incorrect = n - _correct - _abstained
        if _incorrect < 0:
            _incorrect = 0
            _abstained = n - _correct
        _outcome_data.append({'arch': _arch_short[arch], 'Correct': _correct,
                              'Incorrect': _incorrect, 'Abstained': _abstained})

    _odf = pd.DataFrame(_outcome_data).set_index('arch')
    _odf_pct = _odf.div(_odf.sum(axis=1), axis=0) * 100
    _odf_pct.plot(kind='bar', stacked=True, ax=ax5, color=['#2ecc71', '#e74c3c', '#95a5a6'],
                  edgecolor='black', linewidth=0.5)
    ax5.set_ylabel('Percentage of Questions', fontsize=10)
    ax5.set_xlabel('')
    ax5.set_ylim(0, 105)
    ax5.legend(fontsize=10, loc='upper right')
    ax5.set_xticklabels(ax5.get_xticklabels(), rotation=0)
    for i, (_, row) in enumerate(_odf.iterrows()):
        cumsum = 0
        for col in _odf.columns:
            pct = row[col] / row.sum() * 100
            if pct > 8:
                ax5.text(i, cumsum + pct / 2, f"{int(row[col])}",
                         ha='center', va='center', fontsize=9, fontweight='bold')
            cumsum += pct
    fig5.tight_layout()
    display(fig5)
    plt.close(fig5)

    # ==================================================================
    # FIGURE 6: Dynamic Mechanism Behaviour
    # ==================================================================
    if 'dynamic_filtered_results_df' in dir() and len(dynamic_filtered_results_df) > 0:
        _dyn_df = dynamic_filtered_results_df.copy()
    else:
        _dyn_df = _df[_df['architecture'] == 'DYNAMIC_FILTERED_MAS_RAG'].copy()
    if len(_dyn_df) > 0 and 'num_active_agents' in _dyn_df.columns:
        fig6, axes6 = plt.subplots(1, 3, figsize=(15, 5))
        fig6.suptitle('Figure 6: Dynamic Mechanism Behaviour', fontsize=12, y=1.02)

        # Panel A: Active agents by difficulty
        ax = axes6[0]
        for diff in _diff_order:
            sub = _dyn_df[_dyn_df['difficulty'] == diff]
            if len(sub) > 0:
                ax.scatter([diff] * len(sub), sub['num_active_agents'],
                           color=_diff_colors[diff], s=60, edgecolors='black', linewidth=0.5, zorder=5)
                ax.scatter([diff], [sub['num_active_agents'].mean()], color=_diff_colors[diff],
                           s=200, marker='D', edgecolors='black', linewidth=1.5, zorder=10)
        ax.set_ylabel('Active Agents', fontsize=10)
        ax.set_title('Active Agents by Difficulty\n(diamonds = mean)', fontsize=10)
        ax.set_ylim(1.5, 5.5)

        # Panel B: Total tokens vs active agents
        ax = axes6[1]
        for diff in _diff_order:
            sub = _dyn_df[_dyn_df['difficulty'] == diff]
            ax.scatter(sub['num_active_agents'], sub['total_tokens'], color=_diff_colors[diff],
                       s=60, edgecolors='black', linewidth=0.5, label=diff)
        ax.set_xlabel('Active Agents', fontsize=10)
        ax.set_ylabel('Total Tokens', fontsize=10)
        ax.set_title('Tokens by Agent Count', fontsize=10)
        ax.legend(fontsize=8, title='Difficulty')

        # Panel C: Retained evidence by difficulty
        _ev_col = 'retained_evidence_count' if 'retained_evidence_count' in _dyn_df.columns else None
        ax = axes6[2]
        if _ev_col:
            for diff in _diff_order:
                sub = _dyn_df[_dyn_df['difficulty'] == diff]
                if len(sub) > 0:
                    ax.scatter([diff] * len(sub), sub[_ev_col],
                               color=_diff_colors[diff], s=60, edgecolors='black', linewidth=0.5)
                    ax.scatter([diff], [sub[_ev_col].mean()], color=_diff_colors[diff],
                               s=200, marker='D', edgecolors='black', linewidth=1.5, zorder=10)
            ax.set_ylabel('Retained Evidence Chunks', fontsize=10)
            ax.set_title('Retained Context by Difficulty', fontsize=10)
        else:
            ax.text(0.5, 0.5, 'No evidence data\n(run with new KPIs)', ha='center', va='center',
                    transform=ax.transAxes, fontsize=11)
            ax.set_title('Retained Context (N/A)', fontsize=10)

        fig6.tight_layout()
        display(fig6)
        plt.close(fig6)
    else:
        print("Figure 6 skipped: no DYNAMIC results with num_active_agents.")

    # ==================================================================
    # FIGURE 7: Efficiency Analysis (Tokens per Correct Answer)
    # ==================================================================
    fig7, (ax7a, ax7b) = plt.subplots(1, 2, figsize=(13, 5))
    fig7.suptitle('Figure 7: Efficiency Analysis', fontsize=12, y=1.02)

    _f1_col_eff = 'claim_f1' if 'claim_f1' in _df.columns else 'answer_f1'
    _tpc = []
    for arch in _arch_order:
        adf = _df[_df['architecture'] == arch]
        total_tok = adf['total_tokens'].sum()
        n_correct = (adf[_f1_col_eff] > 0).sum()
        tpc = total_tok / max(n_correct, 1)
        _tpc.append({'arch': _arch_short[arch], 'tokens_per_correct': tpc,
                     'total_tokens': total_tok, 'n_correct': n_correct})
    _tpc_df = pd.DataFrame(_tpc)
    bars = ax7a.bar(_tpc_df['arch'], _tpc_df['tokens_per_correct'],
                    color=[_colors[a] for a in _arch_order], edgecolor='black', linewidth=0.5)
    for bar, row in zip(bars, _tpc):
        ax7a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                  f"n={row['n_correct']}", ha='center', fontsize=9)
    ax7a.set_ylabel('Tokens per Correct Answer', fontsize=10)
    ax7a.set_title('Cost per Correct Answer\n(total tokens / correct answers)', fontsize=10)

    # Panel B: Common-correct subset comparison
    if 'MAS_RAG' in _df['architecture'].values and 'DYNAMIC_FILTERED_MAS_RAG' in _df['architecture'].values:
        _mas_eff = _df[_df['architecture'] == 'MAS_RAG'].set_index('question_id')
        _dyn_eff = _df[_df['architecture'] == 'DYNAMIC_FILTERED_MAS_RAG'].set_index('question_id')
        _common_eff = _mas_eff.index.intersection(_dyn_eff.index)
        _both_correct = [q for q in _common_eff
                         if (_mas_eff.loc[q, _f1_col_eff] or 0) > 0 and (_dyn_eff.loc[q, _f1_col_eff] or 0) > 0]
        if _both_correct:
            _mas_tok = [_mas_eff.loc[q, 'total_tokens'] for q in _both_correct]
            _dyn_tok = [_dyn_eff.loc[q, 'total_tokens'] for q in _both_correct]
            x = np.arange(len(_both_correct))
            w = 0.35
            ax7b.bar(x - w/2, _mas_tok, w, label='MAS_RAG', color=_colors['MAS_RAG'],
                     edgecolor='black', linewidth=0.5)
            ax7b.bar(x + w/2, _dyn_tok, w, label='DYN_FILT', color=_colors['DYNAMIC_FILTERED_MAS_RAG'],
                     edgecolor='black', linewidth=0.5)
            ax7b.set_xticks(x)
            ax7b.set_xticklabels(_both_correct, rotation=45, ha='right', fontsize=8)
            ax7b.set_ylabel('Total Tokens', fontsize=10)
            ax7b.set_title(f'Common-Correct Subset (n={len(_both_correct)})\n(questions correct by BOTH)', fontsize=10)
            ax7b.legend(fontsize=9)
            _saving = (sum(_mas_tok) - sum(_dyn_tok)) / sum(_mas_tok) * 100
            ax7b.text(0.5, 0.95, f'DYN saves {_saving:.1f}% tokens on shared successes',
                      transform=ax7b.transAxes, ha='center', va='top', fontsize=9, style='italic')
        else:
            ax7b.text(0.5, 0.5, 'No common-correct questions', ha='center', va='center',
                      transform=ax7b.transAxes, fontsize=11)
    else:
        ax7b.text(0.5, 0.5, 'Need MAS_RAG + DYNAMIC', ha='center', va='center',
                  transform=ax7b.transAxes, fontsize=11)

    fig7.tight_layout()
    display(fig7)
    plt.close(fig7)

    # ==================================================================
    # FIGURE 8: Unanswerable Questions — Abstention Performance
    # ==================================================================
    # This figure evaluates Hallucination Resistance: how well each
    # architecture detects questions that CANNOT be answered from the
    # schema. A stacked bar shows correct abstentions (green) vs
    # hallucinated answers (red) per architecture.
    # ==================================================================
    if len(_unans_df) > 0:
        _unans_archs = [a for a in ['SAS', 'SAS_RAG', 'MAS_RAG', 'DYNAMIC_FILTERED_MAS_RAG'] if a in _unans_df['architecture'].values]
        _n_unans_q = _unans_df['question_id'].nunique()

        fig8, (ax8a, ax8b) = plt.subplots(1, 2, figsize=(14, 5),
                                           gridspec_kw={'width_ratios': [1, 1.5]})
        fig8.suptitle(f'Figure 8: Hallucination Resistance — Unanswerable Questions (N={_n_unans_q})',
                      fontsize=12, y=1.02)

        # Panel A: Stacked bar — correct abstention vs hallucination
        _abs_data = []
        for arch in _unans_archs:
            adf = _unans_df[_unans_df['architecture'] == arch]
            n = len(adf)
            n_abs = int(adf['abstained'].sum()) if 'abstained' in adf.columns else 0
            _abs_data.append({
                'arch': _arch_short.get(arch, arch),
                'Correct Abstention': n_abs,
                'Hallucinated Answer': n - n_abs,
            })
        _abs_df = pd.DataFrame(_abs_data).set_index('arch')
        _abs_df.plot(kind='bar', stacked=True, ax=ax8a,
                     color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.5)
        ax8a.set_ylabel('Number of Questions', fontsize=10)
        ax8a.set_xlabel('')
        ax8a.set_ylim(0, _n_unans_q + 0.8)
        ax8a.set_title('Abstention Rate per Architecture', fontsize=11)
        ax8a.legend(fontsize=9, loc='upper right')
        ax8a.set_xticklabels(ax8a.get_xticklabels(), rotation=0)
        # Annotate counts
        for i, (_, row) in enumerate(_abs_df.iterrows()):
            cumsum = 0
            for col in _abs_df.columns:
                val = row[col]
                if val > 0:
                    ax8a.text(i, cumsum + val / 2, f"{int(val)}",
                              ha='center', va='center', fontsize=11, fontweight='bold',
                              color='white' if val > 1 else 'black')
                cumsum += val
            rate = row['Correct Abstention'] / (row['Correct Abstention'] + row['Hallucinated Answer']) * 100
            ax8a.text(i, cumsum + 0.15, f"{rate:.0f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')

        # Panel B: Per-question binary heatmap
        _unans_questions = sorted(_unans_df['question_id'].unique())
        _heatmap_data = []
        for arch in _unans_archs:
            row = []
            for qid in _unans_questions:
                match = _unans_df[(_unans_df['question_id'] == qid) & (_unans_df['architecture'] == arch)]
                if len(match) > 0 and match.iloc[0].get('abstained', False):
                    row.append(1)  # correct abstention
                elif len(match) > 0:
                    row.append(0)  # hallucinated
                else:
                    row.append(np.nan)
            _heatmap_data.append(row)
        _hm_df = pd.DataFrame(_heatmap_data,
                              index=[_arch_short.get(a, a) for a in _unans_archs],
                              columns=_unans_questions)

        from matplotlib.colors import ListedColormap
        _cmap = ListedColormap(['#e74c3c', '#2ecc71'])
        sns.heatmap(_hm_df, annot=_hm_df.replace({1: '\u2713', 0: '\u2717', np.nan: '\u2014'}).values,
                    fmt='', cmap=_cmap, vmin=0, vmax=1, linewidths=1, linecolor='white',
                    cbar=False, ax=ax8b, annot_kws={'fontsize': 14, 'fontweight': 'bold'})
        ax8b.set_title('Per-Question Abstention Matrix\n(\u2713=correct abstention, \u2717=hallucinated)', fontsize=11)
        ax8b.set_ylabel('')
        ax8b.set_xlabel('')

        fig8.tight_layout()
        display(fig8)
        plt.close(fig8)
    else:
        print("Figure 8 skipped: no unanswerable questions in this run.")

    print("\n" + "\u2501"*70)
    print("\u2713 All thesis figures generated (7 quality + 1 abstention)")
    print("\u2501"*70)